# ResNet-50 classification across the full VisA dataset
Train one shared normal/defective classifier across all 12 product categories using frozen pretrained ResNet-50 features. Normal = 0, defective = 1.

In VS Code Remote SSH, choose BigMac's dl_gpu kernel and run cells from top to bottom. Required libraries are torch, torchvision, pandas, scikit-learn, matplotlib, Pillow, and ipykernel. GPU_INDEX selects one GPU. Image features are extracted once per split and reused.

Use the official high-shot train/test split. Hold out 20% of the training split for validation, stratified by product and label. Compare learning rates [0.0001, 0.001], epochs [10, 100], weighting [off, on], and threshold [0.5, validation-selected]. This produces 16 comparisons from four training runs.

Select the model and one shared threshold using pooled validation results. Evaluate the selected model on the official test split and report each product separately. Validation recall targets do not guarantee test recall. Previous candle test inspection makes the candle portion exploratory.


## 1. Imports and settings

In [ ]:
import csv
import shutil
from itertools import product
from sklearn.metrics import confusion_matrix
import subprocess
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.models import ResNet50_Weights, resnet50
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay,
    PrecisionRecallDisplay, RocCurveDisplay,
)

SEED = 42
GPU_INDEX = 0
BATCH_SIZE = 16
EPOCH_OPTIONS = [10, 100]
LEARNING_RATES = [1e-4, 1e-3]
CLASS_WEIGHT_OPTIONS = [False, True]
THRESHOLD_OPTIONS = ["fixed", "validation_recall"]
VALIDATION_FRACTION = 0.2
TARGET_RECALL = 0.9
THRESHOLD = 0.5
DATA_DIR = Path("visa_data")
OUTPUT_DIR = Path("results/full_visa_resnet50")

torch.manual_seed(SEED)
if torch.cuda.is_available():
    assert GPU_INDEX < torch.cuda.device_count(), "Choose an available GPU index."
    device = torch.device(f"cuda:{GPU_INDEX}")
    for index in range(torch.cuda.device_count()):
        print(index, torch.cuda.get_device_name(index))
else:
    device = torch.device("cpu")
print("Using", device)


## 2. Load all VisA products
Reuse the existing archive and images. Download the approximately 1.9 GB archive if necessary and extract missing images referenced by the official split. The table shows normal/defective counts for each product and split.


In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

def download(url, destination):
    if destination.is_file():
        return
    partial = destination.with_suffix(destination.suffix + ".part")
    subprocess.run([
        "curl", "--fail", "--location", "--retry", "2",
        "--output", str(partial), url,
    ], check=True)
    partial.replace(destination)

split_file = DATA_DIR / "2cls_highshot.csv"
download("https://raw.githubusercontent.com/amazon-science/spot-diff/main/split_csv/2cls_highshot.csv", split_file)
with split_file.open(newline="") as file:
    rows = list(csv.DictReader(file))
assert rows, "No images found for the selected product."

missing_images = {row["image"] for row in rows if not (DATA_DIR / row["image"]).is_file()}
if missing_images:
    archive = DATA_DIR / "VisA_20220922.tar"
    download("https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar", archive)
    with tarfile.open(archive) as tar:
        for member in tar:
            relative_path = Path(member.name)
            if str(relative_path) in missing_images and member.isfile():
                destination = (DATA_DIR / relative_path).resolve()
                assert DATA_DIR.resolve() in destination.parents
                destination.parent.mkdir(parents=True, exist_ok=True)
                with tar.extractfile(member) as source, destination.open("wb") as target:
                    shutil.copyfileobj(source, target)
assert all((DATA_DIR / row["image"]).is_file() for row in rows), "Some images are missing."
table = pd.DataFrame(rows)
display(pd.crosstab([table["object"], table["split"]], table["label"]))


## 3. Prepare images and frozen features
The pretrained weights provide the image resizing and normalization. The encoder stays in evaluation mode to keep its batch normalization statistics fixed.

In [ ]:
weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()

class VisaDataset(Dataset):
    def __init__(self, split):
        self.examples = [row for row in rows if row["split"] == split]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        row = self.examples[index]
        with Image.open(DATA_DIR / row["image"]) as image:
            pixels = preprocess(image.convert("RGB"))
        label = torch.tensor({"normal": 0, "anomaly": 1}[row["label"]], dtype=torch.float32)
        return pixels, label

train_data = VisaDataset("train")
test_data = VisaDataset("test")
assert len(train_data) and len(test_data)
assert {r["image"] for r in train_data.examples}.isdisjoint(r["image"] for r in test_data.examples)

model = resnet50(weights=weights)
model.requires_grad_(False)
feature_count = model.fc.in_features
model.fc = nn.Identity()
encoder = model.to(device).eval()
classifier = nn.Linear(feature_count, 1).to(device)

@torch.no_grad()
def extract_features(dataset):
    features, labels = [], []
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    for images, targets in loader:
        features.append(encoder(images.to(device)).cpu())
        labels.append(targets)
    return TensorDataset(torch.cat(features), torch.cat(labels))

train_features = extract_features(train_data)
train_loader = DataLoader(train_features, batch_size=BATCH_SIZE, shuffle=True)
print("Training features", train_features.tensors[0].shape)


## 4. Compare individual changes and combinations
Compare the baseline with rows that change one setting. Use the full table to inspect interactions. Accuracy should be considered alongside missed defects and false alarms. Threshold changes affect decisions but leave average precision and ROC AUC unchanged for a given checkpoint.

Selection favors the fewest validation false alarms among runs reaching at least 90% recall, then higher average precision and fewer epochs. Training loss scales differ between weighted and unweighted runs.


In [ ]:
# Every experiment uses the same split, initialization, and batch ordering.
features, labels = train_features.tensors
train_indices, validation_indices = train_test_split(
    range(len(labels)), test_size=VALIDATION_FRACTION,
    stratify=[row["object"] + "_" + row["label"] for row in train_data.examples],
    random_state=SEED,
)
fit_features = features[train_indices].to(device)
fit_labels = labels[train_indices].to(device)
validation_features = features[validation_indices].to(device)
validation_labels = labels[validation_indices].numpy().astype(int)
class_ratio = (fit_labels == 0).sum() / (fit_labels == 1).sum()
print("Fit examples:", len(train_indices), "Validation examples:", len(validation_indices))

experiment_rows = []
history_rows = []
model_states = {}

# Train each learning-rate/weighting pair once. Save exact epoch checkpoints.
for learning_rate, weighted in product(LEARNING_RATES, CLASS_WEIGHT_OPTIONS):
    torch.manual_seed(SEED)
    classifier = nn.Linear(feature_count, 1).to(device)
    generator = torch.Generator().manual_seed(SEED)
    fit_loader = DataLoader(
        TensorDataset(fit_features, fit_labels), batch_size=BATCH_SIZE,
        shuffle=True, generator=generator,
    )
    positive_weight = class_ratio if weighted else torch.tensor(1.0, device=device)
    loss_function = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    optimizer = torch.optim.AdamW(classifier.parameters(), lr=learning_rate, weight_decay=1e-4)
    for epoch in range(1, max(EPOCH_OPTIONS) + 1):
        classifier.train()
        total_loss = 0.0
        for batch_features, batch_labels in fit_loader:
            optimizer.zero_grad()
            logits = classifier(batch_features).squeeze(1)
            loss = loss_function(logits, batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_labels)
        history_rows.append({
            "learning_rate": learning_rate, "weighted": weighted,
            "epoch": epoch, "train_loss": total_loss / len(train_indices),
        })
        if epoch not in EPOCH_OPTIONS:
            continue
        classifier.eval()
        with torch.no_grad():
            scores = classifier(validation_features).squeeze(1).sigmoid().cpu().numpy()
        checkpoint = (learning_rate, weighted, epoch)
        model_states[checkpoint] = {
            name: value.detach().cpu().clone()
            for name, value in classifier.state_dict().items()
        }
        for threshold_mode in THRESHOLD_OPTIONS:
            threshold = 0.5
            if threshold_mode == "validation_recall":
                # The highest threshold meeting the recall target minimizes false alarms.
                candidates = sorted(set(scores), reverse=True)
                threshold = next(
                    float(value) for value in candidates
                    if recall_score(validation_labels, scores >= value) >= TARGET_RECALL
                )
            predicted = (scores >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(validation_labels, predicted, labels=[0, 1]).ravel()
            experiment_rows.append({
                "learning_rate": learning_rate, "weighted": weighted,
                "epochs": epoch, "threshold_mode": threshold_mode,
                "threshold": threshold,
                "accuracy": accuracy_score(validation_labels, predicted),
                "recall": recall_score(validation_labels, predicted, zero_division=0),
                "precision": precision_score(validation_labels, predicted, zero_division=0),
                "average_precision": average_precision_score(validation_labels, scores),
                "roc_auc": roc_auc_score(validation_labels, scores),
                "detected_defects": tp, "missed_defects": fn, "false_alarms": fp,
            })
        print(f"Finished lr={learning_rate:g}, weighted={weighted}, epoch={epoch}")

experiments = pd.DataFrame(experiment_rows)
# Count changes relative to the reference 10-epoch, unweighted, 1e-4, 0.5 setup.
experiments["changes"] = (
    experiments.learning_rate.ne(1e-4).astype(int)
    + experiments.weighted.astype(int)
    + experiments.epochs.ne(10).astype(int)
    + experiments.threshold_mode.ne("fixed").astype(int)
)
print("Baseline and one-change experiments (validation results)")
display(experiments[experiments.changes <= 1].sort_values("changes").round(4))
print("All combinations (validation results)")
display(experiments.sort_values(["missed_defects", "false_alarms"]).round(4))

# Select among runs meeting the recall target, favoring fewer false alarms.
eligible = experiments[experiments.recall >= TARGET_RECALL]
selected = eligible.sort_values(
    ["false_alarms", "average_precision", "epochs"], ascending=[True, False, True],
).iloc[0]
best_learning_rate = float(selected.learning_rate)
best_epoch = int(selected.epochs)
selected_weighted = bool(selected.weighted)
THRESHOLD = float(selected.threshold)
positive_weight = class_ratio if selected_weighted else torch.tensor(1.0, device=device)
classifier.load_state_dict(model_states[(best_learning_rate, selected_weighted, best_epoch)])
classifier.eval()
history = pd.DataFrame(history_rows)
print("Selected experiment for test evaluation")
display(selected.to_frame("Selected value"))

fig, axis = plt.subplots(figsize=(9, 4))
for (learning_rate, weighted), run in history.groupby(["learning_rate", "weighted"]):
    axis.plot(run.epoch, run.train_loss, label=f"lr={learning_rate:g}, weighted={weighted}")
axis.set(xlabel="Epoch", ylabel="Training loss")
axis.legend()
plt.show()


## 5. Evaluate the test set
Recall measures how many defects we catch. Precision measures how often a predicted defect is correct. Average precision and ROC AUC evaluate the probability ranking across thresholds.

In [ ]:
test_features = extract_features(test_data)
classifier.eval()
with torch.no_grad():
    probability_batches = []
    for batch_features, _ in DataLoader(test_features, batch_size=BATCH_SIZE):
        probability_batches.append(classifier(batch_features.to(device)).squeeze(1).sigmoid().cpu())
    probabilities = torch.cat(probability_batches).numpy()
actual = test_features.tensors[1].numpy().astype(int)
predicted = (probabilities >= THRESHOLD).astype(int)
assert len(set(actual)) == 2, "Both classes are needed for ROC AUC."
metrics = {
    "accuracy": accuracy_score(actual, predicted),
    "average_precision": average_precision_score(actual, probabilities),
    "roc_auc": roc_auc_score(actual, probabilities),
    "precision": precision_score(actual, predicted, zero_division=0),
    "recall": recall_score(actual, predicted, zero_division=0),
    "f1": f1_score(actual, predicted, zero_division=0),
}
display(pd.DataFrame([metrics], index=["ResNet-50"]).round(4))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(actual, predicted, labels=[0, 1], display_labels=["Normal", "Defective"], ax=axes[0], colorbar=False)
PrecisionRecallDisplay.from_predictions(actual, probabilities, ax=axes[1])
RocCurveDisplay.from_predictions(actual, probabilities, ax=axes[2])
plt.tight_layout()
plt.show()

predictions = pd.DataFrame({
    "image": [row["image"] for row in test_data.examples],
    "product": [row["object"] for row in test_data.examples],
    "label": actual, "probability": probabilities, "prediction": predicted,
})
for title, mistakes in [
    ("Missed defects", predictions[(predictions.label == 1) & (predictions.prediction == 0)]),
    ("False alarms", predictions[(predictions.label == 0) & (predictions.prediction == 1)]),
]:
    print(title, len(mistakes))
    for row in mistakes.head(3).itertuples():
        with Image.open(DATA_DIR / row.image) as image:
            plt.figure(figsize=(3, 3))
            plt.imshow(image)
            plt.title(f"Defect probability {row.probability:.3f}")
            plt.axis("off")
            plt.show()


## 6. Save results and evaluate each product
Save results in results/full_visa_resnet50. Each product has its own counts and metrics so the overall result does not hide weak categories. Running this cell again replaces this experiment's result files.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
experiments.to_csv(OUTPUT_DIR / "validation_experiments.csv", index=False)
history.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
torch.save({
    "classifier_state_dict": classifier.state_dict(),
    "weights": str(weights), "threshold": THRESHOLD,
    "seed": SEED, "epochs": best_epoch, "learning_rate": best_learning_rate, "selected_epoch": best_epoch,
    "validation_fraction": VALIDATION_FRACTION, "target_recall": TARGET_RECALL,
    "positive_weight": float(positive_weight.item()),
}, OUTPUT_DIR / "classifier.pt")
product_rows = []
for product_name, group in predictions.groupby("product"):
    tn, fp, fn, tp = confusion_matrix(group.label, group.prediction, labels=[0, 1]).ravel()
    product_rows.append({
        "product": product_name, "images": len(group),
        "detected_defects": tp, "missed_defects": fn, "false_alarms": fp,
        "accuracy": accuracy_score(group.label, group.prediction),
        "recall": recall_score(group.label, group.prediction, zero_division=0),
        "precision": precision_score(group.label, group.prediction, zero_division=0),
        "average_precision": average_precision_score(group.label, group.probability),
        "roc_auc": roc_auc_score(group.label, group.probability),
    })
product_metrics = pd.DataFrame(product_rows)
display(product_metrics.round(4))
product_metrics.to_csv(OUTPUT_DIR / "per_product_metrics.csv", index=False)
print("Saved results to", OUTPUT_DIR.resolve())
